# Tensors and the Shape of a Network — Hands-on Tutorial

In this notebook you will build intuition for:
1. What a **tensor** is — rank, shape, dtype, and the all-important batch dimension
2. How stacking many neurons turns a weight *vector* into a weight *matrix*
3. How a two-layer network chains shapes — and why two differently-shaped weight matrices still collapse to a single scalar loss
4. Why an architecture like **U-Net** is, at heart, a story about shapes

## Where this fits

| # | Topic | Slide deck | Notebook |
|---|---|---|---|
| 1 | Single neuron | `01_single_neuron.pdf` | `01_single_neuron.ipynb` |
| **→ 1b** | **Tensors & network shape** | *(this notebook)* | **`01b_tensors.ipynb`** |
| 2 | Multilayer networks | `02_multilayer_networks.pdf` | `02_training_loop.ipynb` |
| 3 | Backpropagation | `03_backprop_training.pdf` | `03_backpropagation.ipynb` |
| 4 | Optimizers | `07_optimizers.pdf` | `04_optimizers.ipynb` |
| 5 | CNNs | `04_cnns.pdf` | `05_cnns.ipynb` |
| 6 | Modern architectures | `05a_attention.pdf` + `05b_practical.pdf` | *(bonus: `bonus_generative_models.ipynb`)* |
| 7 | Bayesian inference | `06_bayesian_inference.pdf` | `06_bayesian_inference.ipynb` |

**Coming from:** Notebook 01 computed one neuron's output, $\hat{y} = \sigma(\mathbf{w}\cdot\mathbf{x} + b)$, for a single input vector.

**Leading to:** Notebook 02 trains networks with expressions like `X_train @ w + b` and `nn.Sequential(...)`. This notebook explains where those shapes come from, so none of it looks like magic.

**If you skipped ahead:** You need the neuron equation — weight times input plus bias, through an activation — and comfort reading `numpy`. Nothing more.

---

In [ ]:
# Run on Colab only — skip on JupyterHub (packages are pre-installed)
import sys
if 'google.colab' in sys.modules:
    %pip install -q ipywidgets torch

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider, Dropdown

%matplotlib inline
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(7)
np.random.seed(7)

print(f"PyTorch {torch.__version__}   device = {device}")

## PyTorch in one cell

PyTorch is "NumPy that runs on a GPU and remembers how to differentiate itself." If you know NumPy arrays, you already know 90% of `torch.Tensor`.

Everything in this notebook — data, weights, images, gradients — is a `torch.Tensor`. Four things you do with them constantly:

- **create** — `torch.tensor(...)`, `torch.zeros`, `torch.randn`, `torch.arange`, `torch.from_numpy`
- **inspect** — `.shape`, `.dtype`, `.device` (the three questions to ask of any tensor)
- **operate** — elementwise math (`+ - * /`), reductions (`.sum()`, `.mean()`), and the matrix multiply `@`
- **index / convert** — slice like NumPy, `.numpy()` to go back, `.item()` to pull out a single number, `.to(device)` to move to the GPU

The cell below is a tour of exactly those. Read the printed shapes — that habit is the whole point of this notebook.

In [ ]:
# ── PyTorch in one cell: create, inspect, operate, index, convert ──────────

# CREATE tensors several ways
a = torch.tensor([1.0, 2.0, 3.0])      # from a Python list
z = torch.zeros(2, 3)                   # all zeros, shape (2,3)
r = torch.randn(2, 3)                   # random normal, shape (2,3)
n = torch.arange(0, 6)                  # like range(): [0,1,2,3,4,5]

# INSPECT the three things you always want to know
print(f"r.shape = {tuple(r.shape)}   r.dtype = {r.dtype}   r.device = {r.device}")

# ELEMENTWISE ops act position-by-position; scalars broadcast to every element
print(f"\na + 10   = {(a + 10).tolist()}")
print(f"a * a    = {(a * a).tolist()}     (elementwise, NOT a dot product)")
print(f"a.sum()  = {a.sum().item()}   a.mean() = {a.mean().item():.3f}")

# INDEX and SLICE just like numpy
M = torch.arange(12).reshape(3, 4)
print(f"\nM =\n{M}")
print(f"M[0]     (row 0)      = {M[0].tolist()}")
print(f"M[:, 1]  (column 1)   = {M[:, 1].tolist()}")

# CONVERT to/from numpy and pull scalars out
print(f"\nto numpy : {a.numpy()}")
print(f".item()  : {a[0].item()}   (a Python float, for printing/logging)")

# MOVE to the compute device (CPU here, GPU on Colab if enabled)
print(f"\non device: {a.to(device).device}")

## The linear algebra you actually need

Neural networks use a surprisingly small slice of linear algebra. Three operations carry almost everything:

| Operation | Shapes | What it *is* in a network |
|-----------|--------|---------------------------|
| **Dot product** $\mathbf{w}\cdot\mathbf{x}$ | $(n,)\cdot(n,) \to$ scalar | one neuron |
| **Matrix × vector** $W\mathbf{x}$ | $(H,n)\times(n,) \to (H,)$ | one input through a layer of $H$ neurons |
| **Matrix × matrix** $XW$ | $(N,n)\times(n,H) \to (N,H)$ | a whole **batch** through a layer |

Plus two helpers: **transpose** (swap axes to make the inner dimensions line up) and **reductions** like `sum`, `mean`, `norm` (collapse a tensor to a summary — the loss is a reduction to a single number).

That is genuinely most of it. The dot product is a layer with one neuron; the matrix multiply is that same idea done for every neuron and every example at once.

In [ ]:
# ── The linear algebra you actually need: dot, matrix-vector, matrix-matrix ──

# 1) DOT PRODUCT of two vectors -> a scalar.  This IS a single neuron.
w = torch.tensor([1.5, -0.3])      # (2,)
x = torch.tensor([0.35, 0.12])     # (2,)
print(f"dot  w . x            = {torch.dot(w, x).item():.4f}   (2,)·(2,) -> scalar  [one neuron]")

# 2) MATRIX x VECTOR -> a vector.  One input through a whole layer.
W = torch.tensor([[1.5, -0.3],
                  [0.8,  0.2],
                  [-1.0, 1.1]])    # (3, 2): three neurons
print(f"matvec  W @ x         -> {tuple((W @ x).shape)}   (3,2)@(2,) -> (3,)   [one input, 3 neurons]")

# 3) MATRIX x MATRIX -> a matrix.  A whole BATCH through a whole layer.
Xb = torch.randn(5, 2)             # (5, 2): batch of 5 inputs
print(f"matmul  Xb @ W.T      -> {tuple((Xb @ W.T).shape)}   (5,2)@(2,3) -> (5,3)  [5 inputs, 3 neurons]")

# 4) TRANSPOSE flips which axis is which — needed to line up the inner dims.
print(f"transpose  W {tuple(W.shape)} -> W.T {tuple(W.T.shape)}")

# 5) Useful reductions that turn tensors into summaries:
v = torch.tensor([3.0, 4.0])
print(f"\nnorm  ||v||           = {v.norm().item():.2f}   (length of a vector)")
print(f"sum / mean            = {v.sum().item():.1f} / {v.mean().item():.1f}")

print("\nEvery layer in this notebook is case (3): a batch-matrix times a weight-matrix.")

## A neural network in one picture

Before the details, the whole pipeline in five words: **multiply, add, bend, repeat, score.**

$$\mathbf{x}
\;\xrightarrow[\text{multiply + add}]{\ W_1,\ \mathbf{b}_1\ }\;
\mathbf{z}_1
\;\xrightarrow[\text{bend}]{\ \sigma\ }\;
\mathbf{a}_1
\;\xrightarrow[\text{multiply + add}]{\ W_2,\ \mathbf{b}_2\ }\;
\hat{\mathbf{y}}
\;\xrightarrow[\text{score}]{\ \text{loss}\ }\;
L$$

- **Multiply + add** — a matrix multiply `X @ W + b`. This is *linear*: it can only rotate, scale, and shift.
- **Bend** — a non-linear activation $\sigma$ (ReLU, sigmoid…). Without it, stacking layers collapses back to one linear map. The bend is what buys expressive power.
- **Repeat** — chain as many multiply–bend blocks as you like. "Deep" just means many.
- **Score** — the loss squashes the prediction and the truth into **one scalar** $L$. Training nudges every weight to make $L$ smaller (gradient descent).

Everything below is this picture, made concrete and traced by its shapes. Each of these operations is a **linear-algebra** operation on **tensors** — so we start there.

## Part 0: What is a tensor?

A **tensor** is just an array of numbers with a fixed number of axes. The number of axes is the **rank**; the length along each axis is the **shape**.

| Rank | Name | Example in astronomy | Shape |
|------|------|----------------------|-------|
| 0 | scalar | a single flux measurement | `()` |
| 1 | vector | one galaxy's colours $(g\!-\!r,\ r\!-\!i)$ | `(2,)` |
| 2 | matrix | a **batch** of galaxies, or one grayscale image | `(N, 2)` or `(H, W)` |
| 3 | 3-tensor | a batch of images, or one multi-band image | `(N, H, W)` or `(C, H, W)` |
| 4 | 4-tensor | a batch of multi-band images | `(N, C, H, W)` |

Two ideas do almost all the work in deep learning:

- **The batch dimension.** The first axis, `N`, indexes independent examples. A network processes all `N` at once; the maths for one example is repeated in parallel across the batch. This is why you almost never see a loop over examples — the shape carries it.
- **`dtype` and device.** `float32` on the `cpu` (or `cuda`) is the default currency. Mixing dtypes or devices is the single most common beginner error.

In [ ]:
# ── Build tensors of increasing rank and read off their shapes ──────────────

scalar = torch.tensor(3.14)                       # rank 0
vector = torch.tensor([0.35, 0.12])               # rank 1: one galaxy's (g-r, r-i)
matrix = torch.randn(5, 2)                         # rank 2: a BATCH of 5 galaxies

for name, t in [('scalar', scalar), ('vector', vector), ('matrix', matrix)]:
    print(f"{name:7s}  rank={t.ndim}  shape={tuple(t.shape)}  dtype={t.dtype}")

print()

# ── A single synthetic 'sky image': a Gaussian point source + noise ────────
def make_sky(size=32, n_sources=3, noise=0.05, seed=None):
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:size, 0:size]
    img = np.zeros((size, size), dtype=np.float32)
    for _ in range(n_sources):
        cx, cy = rng.uniform(6, size - 6, 2)
        amp = rng.uniform(0.6, 1.0)
        img += amp * np.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * 2.5 ** 2))
    clean = img.copy()
    noisy = img + rng.normal(0, noise, img.shape).astype(np.float32)
    return clean, noisy

clean, noisy = make_sky(seed=0)
image = torch.from_numpy(noisy)                    # rank 2: one (H, W) image
batch = image.unsqueeze(0).unsqueeze(0)            # rank 4: (N=1, C=1, H, W)

print(f"one image     shape={tuple(image.shape)}   (H, W)")
print(f"as a batch    shape={tuple(batch.shape)}   (N, C, H, W)  <- what a CNN expects")

fig, ax = plt.subplots(1, 2, figsize=(7, 3.2))
ax[0].imshow(clean, cmap='magma'); ax[0].set_title('clean sky  (H, W)')
ax[1].imshow(noisy, cmap='magma'); ax[1].set_title('noisy sky  (H, W)')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

### Think about it

- `matrix = torch.randn(5, 2)` is a batch of 5 galaxies with 2 features each. Which axis would you index to get the third galaxy? Which axis to get everyone's $r\!-\!i$ colour?
- `unsqueeze(0)` added a length-1 axis at the front. Why does a convolutional network insist on the `(N, C, H, W)` shape even when `N = 1` and `C = 1`?
- A scalar has shape `()`, not `(1,)`. What is the difference between a rank-0 tensor and a rank-1 tensor of length 1, and when might it bite you?

---

## Reshaping: same numbers, different axes

The numbers in a tensor are stored as one flat run in memory. The *shape* is just a lens on that run, so you can re-arrange the axes for free:

- **`reshape` / `flatten`** — regroup the same elements into a new shape, in the same order. `-1` tells PyTorch to infer one axis.
- **`transpose` / `.T`** — swap two axes, which genuinely *reorders* the elements.
- **`unsqueeze` / `squeeze`** — add or remove a length-1 axis (we used `unsqueeze` in Part 0 to add the batch and channel axes).

Two places this shows up constantly: **flattening** an image `(N, C, H, W)` into `(N, features)` to feed a plain network, and the **transpose** hiding inside `nn.Linear`, which stores its weight as `(out, in)` and computes `X @ W.T`.

In [ ]:
# ── Same numbers, different arrangement: reshape, flatten, transpose ────────
a = torch.arange(6)                       # (6,)  ->  [0,1,2,3,4,5]
print(f"start    {tuple(a.shape)}  {a.tolist()}")
print(f"reshape  {tuple(a.reshape(2, 3).shape)}   same 6 numbers, filled row by row")
print(f"flatten  {tuple(a.reshape(-1).shape)}     -1 means 'infer this axis'")

m = torch.tensor([[1., 2.],
                  [3., 4.],
                  [5., 6.]])              # (3, 2)
print(f"\nmatrix        {tuple(m.shape)}")
print(f"transpose m.T {tuple(m.T.shape)}   axes SWAPPED — this reorders the data")

# Flattening an image batch to feed a plain (non-convolutional) network:
# keep the batch axis, collapse everything else into one feature vector.
imgs = torch.randn(4, 1, 32, 32)          # (N, C, H, W)
flat = imgs.reshape(4, -1)                # (N, 1*32*32) = (4, 1024)
print(f"\nimages {tuple(imgs.shape)}  ->  flat {tuple(flat.shape)}   (now ready for nn.Linear)")

# Why nn.Linear(2, 8) stores its weight as (8, 2), not (2, 8):
layer = nn.Linear(2, 8)
print(f"\nnn.Linear(2, 8).weight  {tuple(layer.weight.shape)}   it computes  X @ W.T")
print(f"so  (N,2) @ (2,8) -> (N,8)   — the transpose is why the stored shape looks flipped")

### Think about it

- `reshape` and `flatten` keep the elements in the same order; `transpose` (`.T`) actually reorders them. Why does that distinction matter if you flatten an image *after* transposing it?
- `reshape(4, -1)` let PyTorch infer the last axis as `1024`. What is `-1` computing, and when would it fail?
- `nn.Linear(2, 8)` stores its weight as `(8, 2)` and computes `X @ W.T`. Convince yourself the output is `(N, 8)` either way. Why might the library prefer storing `(out, in)`?

---

## Broadcasting: combining different shapes

Addition, subtraction, and multiplication of tensors do **not** require the shapes to match exactly. **Broadcasting** fills the gap:

> Line the shapes up **from the right**. Along each axis, a length that is `1` (or an axis that is missing entirely) is *stretched* to match the other tensor — without ever copying the data.

This is the mechanism behind every `Z + b` in this notebook: a bias of shape `(H,)` is stretched across all `N` rows of a `(N, H)` activation.

It is also the single most common source of **silent** bugs. A `(N, 1)` prediction minus a `(N,)` target does not error — it broadcasts to `(N, N)` and quietly corrupts the loss. **Use the widget below** to combine two shapes: watch for results that match *neither* input — that is the trap.

In [ ]:
# ── Interactive: pick two shapes and see if / how they broadcast ────────────
# Watch for the case where the result matches NEITHER input — that is the
# silent bug (e.g. (100,1) + (100,) -> (100,100)).
shape_opts = {
    '(5, 8)':   (5, 8),
    '(8,)':     (8,),
    '(5, 1)':   (5, 1),
    '(1, 8)':   (1, 8),
    '(100,)':   (100,),
    '(100, 1)': (100, 1),
    '(3, 4)':   (3, 4),
    '(4,)':     (4,),
}

def try_broadcast(shape_a, shape_b):
    A = torch.zeros(shape_opts[shape_a])
    B = torch.zeros(shape_opts[shape_b])
    print(f"A {shape_a}   +   B {shape_b}")
    try:
        out = tuple((A + B).shape)
    except RuntimeError:
        print("  -> CANNOT broadcast: an axis disagrees and neither side is 1")
        return
    if shape_opts[shape_a] == shape_opts[shape_b]:
        print(f"  -> {out}   (shapes already matched)")
    elif out in (shape_opts[shape_a], shape_opts[shape_b]):
        print(f"  -> {out}   (the smaller shape was stretched to match — the useful case, e.g. + bias)")
    else:
        print(f"  -> {out}   <-- matches NEITHER input!  this is the SILENT-BUG case")
        print("     both sides got stretched; almost never what you meant. squeeze/reshape first.")

interact(try_broadcast,
         shape_a=Dropdown(options=list(shape_opts), value='(100, 1)', description='A'),
         shape_b=Dropdown(options=list(shape_opts), value='(100,)',   description='B'));

### Think about it

- `y_hat - y` with shapes `(100, 1)` and `(100,)` gave `(100, 100)` and **no error**. Why is a silent bug more dangerous than a crash? How would you catch it early?
- Broadcasting never copies the smaller tensor in memory — it just reuses it. Why does that matter for a bias added to a `(10000, 512)` activation?
- When you add a bias `b` of shape `(H,)` to `Z` of shape `(N, H)`, which axis is `b` stretched along — the batch axis or the neuron axis?

---

## Part 1: From one neuron to a layer

In notebook 01 a single neuron took an input vector $\mathbf{x}\in\mathbb{R}^2$ and a weight vector $\mathbf{w}\in\mathbb{R}^2$ and produced one number:

$$z = \mathbf{w}\cdot\mathbf{x} + b \qquad (\text{one weight vector} \to \text{one scalar})$$

A **layer** is just many neurons looking at the *same* input. If we want $H$ neurons, we need $H$ weight vectors — stack them as the columns of a **weight matrix** $W\in\mathbb{R}^{2\times H}$:

$$\mathbf{z} = \mathbf{x}\,W + \mathbf{b} \qquad (\mathbb{R}^{2} \to \mathbb{R}^{H})$$

And because we process a whole **batch** of $N$ inputs at once, $\mathbf{x}$ becomes a matrix $X\in\mathbb{R}^{N\times 2}$:

$$Z = X\,W + \mathbf{b}, \qquad \underbrace{(N,2)}_{X} \times \underbrace{(2,H)}_{W} \to \underbrace{(N,H)}_{Z}$$

**The rule for matrix multiply:** the inner dimensions must match and cancel; the outer dimensions survive. `(N, 2) @ (2, H) -> (N, H)`. The `2` is consumed — it is the number of input features. `N` and `H` remain: `N` batch examples, each now described by `H` neuron outputs.

In [ ]:
# ── Interactive: watch a layer's output shape track the number of neurons ───
# Drag H. The input is fixed (5 galaxies, 2 features); only the weight matrix
# and the output width change.
N, n_features = 5, 2
X_demo = torch.randn(N, n_features)          # a batch of 5 galaxies, 2 colours

def layer_shapes(H):
    W = torch.randn(n_features, H)           # H neurons = H weight vectors
    b = torch.zeros(H)                        # one bias per neuron
    Z = X_demo @ W + b

    print(f"H = {H} neurons")
    print(f"  weight matrix W : {tuple(W.shape)}   ({H} weight vectors of length {n_features})")
    print(f"  bias vector  b  : {tuple(b.shape)}")
    print()
    print(f"  X {tuple(X_demo.shape)}  @  W {tuple(W.shape)}  +  b {tuple(b.shape)}")
    print(f"    ->  Z {tuple(Z.shape)}   ({N} inputs, each now described by {H} numbers)")
    print()
    print(f"  the inner '2' cancels; N={N} and H={H} survive")
    print(f"  parameters here: {n_features*H} weights + {H} biases = {n_features*H + H}")
    print("\n  output width  " + "|" * H + f"  ({H})")

interact(layer_shapes,
         H=IntSlider(value=8, min=1, max=16, step=1, description='neurons H'));

### Think about it

- `nn.Linear(2, 8)` stores its weight with shape `(8, 2)`, the transpose of our `W`. Why? (Hint: it computes `X @ W.T`.) Does the maths change?
- The bias for a layer has shape `(H,)`, not `(1,)`. When it is added to `Z` of shape `(N, H)`, which axis does it broadcast along?
- If you wanted a layer that maps 2 features to 100 neurons, what shape is the weight matrix, and how many parameters (weights + biases) does the layer have?

---

## Part 2: A two-layer network — different shapes, one scalar loss

Now stack two layers with **deliberately different shapes**, on the same photometric-redshift task as notebook 02: predict redshift $z$ from two galaxy colours.

$$\underbrace{X}_{(N,2)} \xrightarrow{\;W_1\,(2,8)\;} \underbrace{H_1}_{(N,8)} \xrightarrow{\text{ReLU}} \underbrace{A_1}_{(N,8)} \xrightarrow{\;W_2\,(8,1)\;} \underbrace{\hat{y}}_{(N,1)} \xrightarrow{\text{MSE}} \underbrace{L}_{()}$$

$W_1$ is `(2, 8)` and $W_2$ is `(8, 1)` — completely different shapes. They chain because the **output width of one layer is the input width of the next**: the `8` produced by layer 1 is exactly the `8` consumed by layer 2. This is the single constraint that makes a network well-formed.

No matter how many layers or how their shapes differ, the final output is reduced by the loss to a **single scalar** `()`. That scalar is what we differentiate. A whole network, however wide or deep, funnels down to one number to minimise.

In [ ]:
# ── Photometric-redshift data (same recipe as notebook 02) ─────────────────
Ng = 200
z_true = np.random.uniform(0.02, 0.40, Ng)
gr = 0.80 * z_true + 0.15 + np.random.normal(0, 0.04, Ng)
ri = 0.40 * z_true + 0.05 + np.random.normal(0, 0.03, Ng)
X_np = np.column_stack([gr, ri]).astype(np.float32)
y_np = z_true.astype(np.float32).reshape(-1, 1)
X = torch.from_numpy(X_np)
y = torch.from_numpy(y_np)

# ── Two weight matrices of DIFFERENT shapes ────────────────────────────────
torch.manual_seed(0)
W1 = torch.randn(2, 8, requires_grad=True) * 0.5   # (2, 8)
b1 = torch.zeros(8, requires_grad=True)
W2 = torch.randn(8, 1, requires_grad=True) * 0.5   # (8, 1)
b2 = torch.zeros(1, requires_grad=True)

# ── Forward pass, printing the shape at every stage ────────────────────────
H1 = X @ W1 + b1          # (N,2) @ (2,8) -> (N,8)
A1 = torch.relu(H1)       # (N,8)  — activation keeps the shape
y_hat = A1 @ W2 + b2      # (N,8) @ (8,1) -> (N,1)
loss = torch.mean((y_hat - y) ** 2)   # (N,1) -> ()  a single scalar

print("SHAPE FLOW THROUGH THE NETWORK")
print(f"  X       {tuple(X.shape)}")
print(f"  @ W1    {str(tuple(W1.shape)):<8} ->  H1  {tuple(H1.shape)}")
print(f"  ReLU                 ->  A1  {tuple(A1.shape)}")
print(f"  @ W2    {str(tuple(W2.shape)):<8} ->  y_hat {tuple(y_hat.shape)}")
print(f"  MSE                  ->  loss {tuple(loss.shape)}   <- one scalar, value = {loss.item():.4f}")
print()
print("Two differently-shaped weight matrices, W1 (2,8) and W2 (8,1),")
print("chain cleanly because 8 (out of layer 1) == 8 (into layer 2),")
print("and everything collapses to a single number we can minimise.")

In [ ]:
# ── Train the two-layer network with those same different-shaped weights ────
torch.manual_seed(0)
W1 = (torch.randn(2, 8) * 0.5).requires_grad_(True)
b1 = torch.zeros(8, requires_grad=True)
W2 = (torch.randn(8, 1) * 0.5).requires_grad_(True)
b2 = torch.zeros(1, requires_grad=True)
params = [W1, b1, W2, b2]

lr = 0.05
losses = []
for epoch in range(2000):
    y_hat = torch.relu(X @ W1 + b1) @ W2 + b2
    loss = torch.mean((y_hat - y) ** 2)
    loss.backward()
    with torch.no_grad():
        for p in params:
            p -= lr * p.grad
            p.grad.zero_()
    losses.append(loss.item())

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(losses, color='tab:blue')
ax[0].set_xlabel('Epoch'); ax[0].set_ylabel('MSE loss'); ax[0].set_yscale('log')
ax[0].set_title('Two-layer network training loss'); ax[0].grid(alpha=0.3)

z_hat = (torch.relu(X @ W1 + b1) @ W2 + b2).detach().numpy()
ax[1].scatter(y_np, z_hat, s=15, alpha=0.6, color='tab:blue')
ax[1].plot([0.02, 0.40], [0.02, 0.40], 'k--', lw=1, label='perfect')
ax[1].set_xlabel('True redshift z'); ax[1].set_ylabel('Predicted redshift')
ax[1].set_title('Model fit'); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Gradients had the SAME shapes as their weights:")
print(f"  dL/dW1 shape == W1 shape == (2, 8)")
print(f"  dL/dW2 shape == W2 shape == (8, 1)")
print(f"Final loss = {losses[-1]:.6f}   (a single scalar drove every update)")

### Think about it

- The gradient `dL/dW1` has the *same shape* as `W1` itself, `(2, 8)`. Why must that always be true, whatever the loss?
- We chose the hidden width `8` arbitrarily. What breaks if you set `W2 = torch.randn(4, 1)` instead of `(8, 1)`? Try it and read the error — it names the mismatched dimension.
- The loss is always a scalar `()`, even though `y_hat` is `(N, 1)`. Where did the `N` go? What would happen to training if the loss were a vector instead?
- Compare this to the linear model in notebook 02. What does the ReLU between the two layers buy you that a single `(2, 1)` weight matrix cannot?

---

## Feature scaling — the preprocessing that shapes cannot show

Shapes tell you whether the maths is *legal*. They say nothing about whether it *converges*. The most common reason a correctly-shaped network refuses to learn is **unscaled inputs**.

Gradient descent takes one shared step size $\eta$ for every weight. But the gradient for a feature scales with that feature's magnitude. If one column is a magnitude in the range 15–22 and another is a colour around 0.3, their gradients differ by orders of magnitude — no single $\eta$ works for both. The loss surface becomes a long, thin valley and the optimiser crawls or diverges.

The fix is **standardisation**: per feature, subtract the mean and divide by the standard deviation, so every input column has mean $\approx 0$ and std $\approx 1$. Same data, same shape, dramatically better conditioning.

In [ ]:
# ── Interactive: feature scale x learning rate — when does training survive? ─
# Left slider: how much bigger feature 0 is. Right slider: the learning rate.
# Green (standardised) should stay robust; red (raw) blows up as scale grows.
import math

def train_linear(Xin, lr, epochs=150):
    torch.manual_seed(0)
    w = torch.zeros(2, 1, requires_grad=True)
    b = torch.zeros(1, requires_grad=True)
    hist = []
    for _ in range(epochs):
        loss = torch.mean((Xin @ w + b - y) ** 2)
        loss.backward()
        with torch.no_grad():
            w -= lr * w.grad; b -= lr * b.grad
            w.grad.zero_(); b.grad.zero_()
        hist.append(loss.item())
    return hist

def scaling_demo(log_scale, log_lr):
    scale = 10 ** log_scale
    lr    = 10 ** log_lr
    X_bad = X.clone(); X_bad[:, 0] = X_bad[:, 0] * scale
    X_std = (X_bad - X_bad.mean(0)) / X_bad.std(0)

    hist_bad = train_linear(X_bad, lr)
    hist_std = train_linear(X_std, lr)

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(hist_bad, 'o-', ms=3, color='tomato',   label=f'raw (feature 0 x{scale:g})')
    ax.plot(hist_std,        color='seagreen',       label='standardised')
    ax.set_yscale('log'); ax.set_ylim(1e-5, 1e6)
    ax.set_xlabel('Epoch'); ax.set_ylabel('MSE loss')
    ax.set_title(f'scale = x{scale:g}     lr = {lr:.4g}')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    raw = 'diverged' if not math.isfinite(hist_bad[-1]) else f'{hist_bad[-1]:.5f}'
    print(f"raw final loss: {raw}      standardised final loss: {hist_std[-1]:.5f}")

interact(scaling_demo,
         log_scale=FloatSlider(value=2.7, min=0.0, max=3.0, step=0.1, description='log10 scale'),
         log_lr=FloatSlider(value=-1.0, min=-4.0, max=-0.5, step=0.1, description='log10 lr'));

### Think about it

- Standardising did not change the *shape* of anything — `X_std` is still `(200, 2)`. So why does it change how fast training converges?
- The gradient for a feature is proportional to that feature's magnitude. If one column is ~500× larger, what happens to the relative size of its gradient, and hence to a shared learning rate?
- We computed `mean` and `std` on the training data. Why must the *validation* and *test* sets be scaled with those **same** numbers, not their own?
- Batch-norm and layer-norm (later lectures) standardise activations *inside* the network for the same reason. What problem are they solving that a one-time input scaling cannot?

---

## Part 3: Shape *is* architecture — a minimal U-Net

So far shapes only got **narrower** (`2 → 8 → 1`). For images, the interesting architectures also change the **spatial** shape. A U-Net — the workhorse for denoising, segmentation, and deconvolution in astronomy — gets its name and its power directly from what it does to `(H, W)`:

```
  input  (1, 32, 32)
     |  encoder: conv, then downsample  (halve H, W;  grow channels)
   (16, 16, 16)
     |
   (32,  8,  8)   <- bottleneck: smallest spatial size, most channels
     |  decoder: upsample  (double H, W;  shrink channels)
   (32, 16, 16) --concat-- skip from encoder (32, 16, 16)
     |
   (16, 32, 32) --concat-- skip from encoder (16, 32, 32)
     |
  output (1, 32, 32)
```

The **“U” is a plot of the spatial size**: it shrinks down the left arm, bottoms out, and grows back up the right arm. The two horizontal `concat` arrows are **skip connections** — they only work because the encoder feature at each level has *exactly the shape* the decoder produces at the matching level. Architecture here is nothing more than a set of shape-matching constraints.

We train it on a real task: **denoise the synthetic sky images** from Part 0.

In [ ]:
# ── A minimal U-Net, written to make the shape story explicit ──────────────
class MiniUNet(nn.Module):
    def __init__(self, ch=16):
        super().__init__()
        # encoder blocks (conv keeps H,W;  MaxPool halves them)
        self.enc1 = nn.Sequential(nn.Conv2d(1, ch, 3, padding=1), nn.ReLU())
        self.enc2 = nn.Sequential(nn.Conv2d(ch, 2*ch, 3, padding=1), nn.ReLU())
        self.pool = nn.MaxPool2d(2)
        # bottleneck
        self.bottleneck = nn.Sequential(nn.Conv2d(2*ch, 4*ch, 3, padding=1), nn.ReLU())
        # decoder: ConvTranspose doubles H,W;  concat skip;  conv mixes them
        self.up2  = nn.ConvTranspose2d(4*ch, 2*ch, 2, stride=2)
        self.dec2 = nn.Sequential(nn.Conv2d(4*ch, 2*ch, 3, padding=1), nn.ReLU())
        self.up1  = nn.ConvTranspose2d(2*ch, ch, 2, stride=2)
        self.dec1 = nn.Sequential(nn.Conv2d(2*ch, ch, 3, padding=1), nn.ReLU())
        self.out  = nn.Conv2d(ch, 1, 1)

    def forward(self, x, trace=False):
        def show(name, t):
            if trace: print(f"  {name:12s} {tuple(t.shape)}")
        show('input', x)
        e1 = self.enc1(x);            show('enc1', e1)          # (N, ch, 32, 32)
        e2 = self.enc2(self.pool(e1)); show('enc2', e2)         # (N, 2ch, 16, 16)
        b  = self.bottleneck(self.pool(e2)); show('bottleneck', b)  # (N, 4ch, 8, 8)
        d2 = self.up2(b);             show('up2', d2)           # (N, 2ch, 16, 16)
        d2 = self.dec2(torch.cat([d2, e2], dim=1)); show('cat+dec2', d2)  # concat -> (N,4ch,16,16) -> (N,2ch,16,16)
        d1 = self.up1(d2);            show('up1', d1)           # (N, ch, 32, 32)
        d1 = self.dec1(torch.cat([d1, e1], dim=1)); show('cat+dec1', d1)  # concat -> (N,2ch,32,32) -> (N,ch,32,32)
        y  = self.out(d1);            show('output', y)         # (N, 1, 32, 32)
        return y

# ── Interactive: change the base channel width and watch the trace + size ───
def unet_trace(ch):
    net = MiniUNet(ch=ch)
    dummy = torch.randn(2, 1, 32, 32)
    print(f"MiniUNet(ch={ch}) — shape trace (batch of 2 images):")
    net(dummy, trace=True)
    n = sum(p.numel() for p in net.parameters())
    print(f"\n  channels:  {ch} -> {2*ch} -> {4*ch} (bottleneck) -> {2*ch} -> {ch}")
    print(f"  spatial :  32 -> 16 -> 8 -> 16 -> 32   <- this path IS the letter U")
    print(f"  total parameters: {n:,}   (grows ~quadratically with ch)")
    print("\n  skip-concat works only because enc2 (16x16) meets up2 (16x16), enc1 (32x32) meets up1 (32x32)")

interact(unet_trace,
         ch=IntSlider(value=16, min=4, max=32, step=4, description='base ch'));

In [ ]:
# ── Train the U-Net to denoise synthetic sky images ────────────────────────
def make_dataset(n, size=32, seed=0):
    rng = np.random.default_rng(seed)
    cleans, noisys = [], []
    for i in range(n):
        c, nz = make_sky(size=size, n_sources=rng.integers(2, 5),
                         noise=0.08, seed=rng.integers(1e9))
        cleans.append(c); noisys.append(nz)
    clean = torch.from_numpy(np.stack(cleans)[:, None])   # (n, 1, H, W)
    noisy = torch.from_numpy(np.stack(noisys)[:, None])
    return noisy, clean

Xtr_noisy, Xtr_clean = make_dataset(256, seed=1)
Xte_noisy, Xte_clean = make_dataset(16,  seed=99)
Xtr_noisy, Xtr_clean = Xtr_noisy.to(device), Xtr_clean.to(device)
print(f"train inputs (noisy) {tuple(Xtr_noisy.shape)}   targets (clean) {tuple(Xtr_clean.shape)}")

net = MiniUNet().to(device)
opt = torch.optim.Adam(net.parameters(), lr=1e-3)
criterion = nn.MSELoss()

losses, bs = [], 32
for epoch in range(40):
    perm = torch.randperm(len(Xtr_noisy))
    epoch_l = []
    for s in range(0, len(perm), bs):
        idx = perm[s:s+bs]
        opt.zero_grad()
        pred = net(Xtr_noisy[idx])
        loss = criterion(pred, Xtr_clean[idx])
        loss.backward(); opt.step()
        epoch_l.append(loss.item())
    losses.append(float(np.mean(epoch_l)))

plt.figure(figsize=(8, 3.5))
plt.plot(losses, color='tab:blue'); plt.yscale('log')
plt.xlabel('Epoch'); plt.ylabel('MSE loss'); plt.title('U-Net denoising loss')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
print(f"Final loss = {losses[-1]:.5f}")

In [ ]:
# ── Before / after: what the trained U-Net does to held-out images ─────────
net.eval()
with torch.no_grad():
    pred = net(Xte_noisy.to(device)).cpu()

fig, axes = plt.subplots(3, 4, figsize=(11, 8))
for col in range(4):
    axes[0, col].imshow(Xte_noisy[col, 0], cmap='magma')
    axes[1, col].imshow(pred[col, 0],      cmap='magma')
    axes[2, col].imshow(Xte_clean[col, 0], cmap='magma')
for ax in axes.ravel(): ax.axis('off')
axes[0, 0].set_ylabel('noisy input',  rotation=90, size=11); axes[0, 0].axis('on'); axes[0, 0].set_xticks([]); axes[0, 0].set_yticks([])
axes[1, 0].set_ylabel('U-Net output', rotation=90, size=11); axes[1, 0].axis('on'); axes[1, 0].set_xticks([]); axes[1, 0].set_yticks([])
axes[2, 0].set_ylabel('clean truth',  rotation=90, size=11); axes[2, 0].axis('on'); axes[2, 0].set_xticks([]); axes[2, 0].set_yticks([])
plt.suptitle('Top: noisy in   |   Middle: U-Net denoised   |   Bottom: truth', y=0.98)
plt.tight_layout(); plt.show()

input_mse = criterion(Xte_noisy, Xte_clean).item()
output_mse = criterion(pred, Xte_clean).item()
print(f"MSE noisy-vs-clean  : {input_mse:.5f}")
print(f"MSE denoised-vs-clean: {output_mse:.5f}   ({input_mse/output_mse:.1f}x lower)")

### Think about it

- Trace the spatial size through `forward`: `32 → 16 → 8 → 16 → 32`. If you added one more encoder/decoder level, what would the bottleneck size be, and what constraint does that place on the input size?
- `torch.cat([d2, e2], dim=1)` concatenates along the **channel** axis. Why `dim=1` and not `dim=0`? What would concatenating along `dim=0` do instead?
- The skip connection feeds the encoder feature `e2` directly to the decoder. What information might survive in `e2` that has been lost by the time the signal reaches the bottleneck?
- The U-Net has far more parameters than the two-layer network in Part 2, yet the loss is still a single scalar. Did anything about *how* we train change between a `(2,8)` weight matrix and a `(32,16,16)` feature map?

---

## Going further

### Part A — Go deeper

**Challenge:** change the hidden width in Part 2 from `8` to `4`, then `32`, then `128`. Keep everything else fixed. Plot final loss vs hidden width. Does more width always help? At what point does the shape stop being the bottleneck and the data (200 noisy galaxies) become the limit?

### Part B — Lead forward

**Challenge:** the U-Net in Part 3 downsamples with `MaxPool2d` and upsamples with `ConvTranspose2d`. Print the shape of `net.up2.weight` and work out, from its shape alone, how `ConvTranspose2d` turns an `8×8` feature map into `16×16`. Then remove **one** skip connection (feed `self.dec1` only `d1`, not `torch.cat([d1, e1], dim=1)` — you will need to change `dec1`'s input channels) and retrain. How much worse is the denoising? This is the experiment that first showed skip connections matter.

---

### References

| | |
|---|---|
| **Video** | 3Blue1Brown — *But what is a neural network?* [youtube.com/watch?v=aircAruvnKk](https://www.youtube.com/watch?v=aircAruvnKk) — the layer-as-matrix picture |
| **Primary** | Ronneberger, Fischer & Brox (2015) — *U-Net: Convolutional Networks for Biomedical Image Segmentation.* MICCAI. [arxiv.org/abs/1505.04597](https://arxiv.org/abs/1505.04597) — the original U-Net |
| **Reference** | PyTorch docs — *Tensor* and *broadcasting semantics* [pytorch.org/docs/stable/tensors.html](https://pytorch.org/docs/stable/tensors.html) |
| **Astronomy** | Many radio/optical pipelines use U-Nets for deconvolution and source finding — the shape story here is exactly the one used at survey scale |